<a href="https://colab.research.google.com/github/alexanderquispe/Diplomado_PUCP/blob/group_4_ass_10_2024/Lecture_14/Assignment_10/group_4_ass_10_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Get the 15 variables from this raster for all Peru departments polygons

## 1.1 Install and import

In [ ]:
!pip install rasterio

In [ ]:
%pip install geopandas matplotlib shapely rasterio numpy pandas sklearn-xarray -q

In [ ]:
!pip install ipywidgets
!jupyter labextension install @jupyter-widgets/jupyterlab-manager

In [ ]:
!pip install rasterstats

In [ ]:
%pip install geopandas matplotlib shapely rasterio numpy pandas sklearn-xarray -q
%pip install git+https://github.com/jgrss/geowombat  -q

In [ ]:
import geopandas as gpd
from rasterstats import zonal_stats
import pandas as pd
import geowombat as gw

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.merge import merge
from rasterio.plot import show
from shapely.geometry import mapping
import rasterio
import rasterio

# import plotting
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

import os
from rasterio.mask import mask

## 1.2. Set up and load

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import geopandas as gpd

# Ruta al shapefile
shapefile_path = '/content/drive/MyDrive/Assign 10/INEI_LIMITE_DEPARTAMENTAL/INEI_LIMITE_DEPARTAMENTAL.shp'

# Leer el shapefile usando Geopandas
departments = gpd.read_file(shapefile_path)

# Transformar el CRS de los departamentos al CRS del raster si es necesario
# Asumiendo que los rasters están en 'esri:54009'
departments = departments.to_crs('esri:54009')

print(departments.head())

In [ ]:
import rasterio
from rasterio.merge import merge
import rasterio
from rasterio.mask import mask
import pandas as pd

In [ ]:
# Rutas a los archivos raster
raster_files = ['/content/drive/MyDrive/Assign 10/tif_files_group4/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C10.tif',
                '/content/drive/MyDrive/Assign 10/tif_files_group4/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C11.tif',
                '/content/drive/MyDrive/Assign 10/tif_files_group4/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C12.tif',
                '/content/drive/MyDrive/Assign 10/tif_files_group4/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R11_C11.tif',
                '/content/drive/MyDrive/Assign 10/tif_files_group4/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R11_C12.tif',
                '/content/drive/MyDrive/Assign 10/tif_files_group4/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R12_C11.tif',
                '/content/drive/MyDrive/Assign 10//tif_files_group4/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R12_C12.tif',
                '/content/drive/MyDrive/Assign 10/tif_files_group4/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R9_C10.tif',
                '/content/drive/MyDrive/Assign 10/tif_files_group4/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R9_C11.tif',
                '/content/drive/MyDrive/Assign 10/tif_files_group4/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R9_C12.tif']

In [ ]:
from rasterstats import zonal_stats
# Inicializar lista para almacenar los resultados de cada raster
all_stats = []
# Iterar sobre cada archivo raster
for raster_path in raster_files:
    # Calcular estadísticas zonales
    stats = zonal_stats(departments, raster_path, stats="count sum", categorical=True, all_touched=True)
    # Convertir estadísticas a DataFrame
    stats_df = pd.DataFrame(stats)
    # Concatenar con la geometría y la información de los departamentos
    df = pd.concat([departments.reset_index(drop=True), stats_df], axis=1)
    all_stats.append(df)

In [ ]:
# Concatenar todos los DataFrames en all_stats en un único DataFrame
final_df = pd.concat(all_stats, ignore_index=True)

# Verificar el resultado
print(final_df.head())
print(f"Total de filas en el DataFrame consolidado: {len(final_df)}")

In [ ]:
# Verificar si hay valores nulos o datos faltantes que necesiten ser tratados
print(final_df.isnull().sum())

In [ ]:
# Reemplazar todos los valores NaN por 0 en todo el DataFrame
final_df.fillna(0, inplace=True)

# Verificar el resultado después del reemplazo
print(final_df.head())

In [ ]:
final_df

Cálculo del porcentaje de cobertura de diferentes MSZ (Zonas de Asentamiento Morfológico) representadas en un raster para cada uno de los polígonos:

In [ ]:
#1. Definition of the pixel area. We establish that each pixel represents 100 ^2 (The spatial raster dataset delineates the boundaries of the human settlements at 10m resolution).

pixel_area = 100

#2. Iteration on each 15 cathegories of MSZ
    #We want to iterate on each 15 cathegories to calculate the cover percentage of each of the geographical areas. For that, we create a new column in final_df (MSZ_[cathegory]cogerage)


for category in range(1, 16):  # Asumiendo 15 categorías de MSZ
    # Usign .apply we apply a function to each row. In each row, we try to obtain the correspondent value to the MSZ (row.get(str(cathegory), 0))
    # (we count the pixels on each row). We then multiply the pixels by the pixel area so we can obtain the total area covered by that MSZ cathegory in M2.
    #We divide that area by the polygon area (row[geometry]area) so we calculate which fraction of the polygon is covered by the MSZ.
 final_df[f'MSZ_{category}_coverage'] = final_df.apply(
        lambda row: ((row.get(str(category), 0) * pixel_area) / row['geometry'].area) * 100 if str(category) in row else 0, #We multiply the result by 100 so we see the percentage
        axis=1
    )

In [ ]:
print(final_df.columns) #We see that we now have a new column for each cathegory of MSZ.

#2. choropleth map using folium for these 15 variables

In [ ]:

!pip install mapclassify

In [ ]:
for i in range(1, 16):
    column_name = f'MSZ_{i}_coverage'
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    final_df.plot(
        column=column_name,
        cmap='YlGnBu',
        legend=True,
        scheme='quantiles',
        ax=ax
    )

    # Manually configure the legend
    leg = ax.get_legend()
    leg.set_title(f'MSZ {i}')
    leg.set_bbox_to_anchor((1.5, 0.5))

    plt.show()

3. Save your html in the same folder of your JN. Name your HTML as your branch. This HTML should have all these layers. Please do not forget to use Layer Control.

In [ ]:
!pip install geopandas folium matplotlib

In [ ]:
import folium
from folium.plugins import MarkerCluster

# We create a base map centered on a specific location
map_center = [-12.0757538, -76.9863174]
m = folium.Map(location=map_center, zoom_start=5)

In [ ]:
# We add layers to the map
for i in range(1, 16):
    column_name = f'MSZ_{i}_coverage'
    geojson_data = final_df[['geometry', column_name]].to_crs("EPSG:4326").to_json()
    folium.Choropleth(
        geo_data=geojson_data,
        data=final_df,
        columns=['geometry', column_name],
        key_on="feature.properties.geometry",
        fill_color='YlGnBu',
        fill_opacity=0.7,
        line_opacity=0.2,
        legend_name=f'MSZ {i}',
    ).add_to(m)

# Add Layer Control
folium.LayerControl().add_to(m)

# We save the map as an HTML file in our branch
m.save('Lecture_14/Assignment_10/group_4_ass_10_2024.html')

